# 訓練済みモデル検証ノートブック

このノートブックでは、`out/` 配下の学習済み run ディレクトリを指定して重みを読み込み、指定したテストデータディレクトリで推論・評価を行います。

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

# リポジトリルート / src を import path に追加
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ctmc_surrogate.data.collate import ctmc_collate_fn
from ctmc_surrogate.data.dataset import CTMCSurrogateDataset
from ctmc_surrogate.data.dataset_csv_loader import ParsedCTMCDataset, load_dir
from ctmc_surrogate.data.dataset_screening import ScreeningConfig, extract_lambdas_from_Q, screen_datasets
from ctmc_surrogate.models import build_model
from ctmc_surrogate.train import CustomLoss


In [ ]:
# ===== ユーザー設定セル（必要に応じて編集してください） =====
# 例: REPO_ROOT / "out" / "run_20260101_120000"
model_run_dir = REPO_ROOT / "out" / "<ここにrunディレクトリ名を入力>"

# データディレクトリは「絶対パス」で指定してください
# 例: Path("/workspace/CTMCxDeepSets_publish/data/test")
test_data_dir = Path("/absolute/path/to/test_data")

# オプション設定
recursive = True
batch_size = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 評価データ用スクリーニング設定
# True の場合、train_entrypoint と同様に λ の閾値・構造・NaN/Inf を用いて除外します
enable_screening = True
screening_min_lambda = 1e-2
screening_max_lambda = 1
screening_check_nan_inf = True
screening_require_structure = True

print(f"model_run_dir: {model_run_dir}")
print(f"test_data_dir: {test_data_dir}")
print(f"device: {device}")
print(
    "screening:",
    {
        "enabled": enable_screening,
        "min_lambda": screening_min_lambda,
        "max_lambda": screening_max_lambda,
        "check_nan_inf": screening_check_nan_inf,
        "require_structure": screening_require_structure,
    },
)


In [ ]:
# out/ 直下の run ディレクトリ候補を表示
out_dir = REPO_ROOT / "out"
if out_dir.exists():
    candidates = sorted([p for p in out_dir.iterdir() if p.is_dir()])
    if len(candidates) == 0:
        print("out/ 配下にディレクトリが見つかりませんでした。")
    else:
        print("利用可能な run ディレクトリ候補:")
        for p in candidates:
            print(f"- {p}")
else:
    print("out/ ディレクトリが存在しません。")

In [ ]:
def read_yaml_like_dict(path: Path) -> dict:
    """train_loop.save_run_artifacts が保存する簡易 YAML 形式を辞書として読む。"""
    config: dict[str, object] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        striped = line.strip()
        if not striped or striped.startswith("#"):
            continue
        if ":" not in striped:
            continue
        key, value = striped.split(":", 1)
        key = key.strip()
        raw = value.strip().strip('\"')
        lowered = raw.lower()
        if lowered in {"true", "false"}:
            config[key] = lowered == "true"
            continue
        try:
            config[key] = int(raw)
            continue
        except ValueError:
            pass
        try:
            config[key] = float(raw)
            continue
        except ValueError:
            pass
        config[key] = raw
    return config


def validate_paths(run_dir: Path, data_dir: Path) -> None:
    if not run_dir.exists():
        raise FileNotFoundError(f"run ディレクトリが存在しません: {run_dir}")
    if not data_dir.is_absolute():
        raise ValueError(f"test_data_dir は絶対パスで指定してください: {data_dir}")
    if not data_dir.exists():
        raise FileNotFoundError(f"test_data_dir が存在しません: {data_dir}")


def apply_eval_screening(datasets: list[ParsedCTMCDataset]) -> list[ParsedCTMCDataset]:
    """評価データに train と同等のスクリーニングを適用し、通過データのみ返す。"""
    cfg = ScreeningConfig(
        min_lambda=float(screening_min_lambda),
        max_lambda=float(screening_max_lambda),
        check_nan_inf=bool(screening_check_nan_inf),
        require_structure=bool(screening_require_structure),
    )
    result = screen_datasets(datasets, cfg)
    print(
        "スクリーニング結果:",
        {
            "入力件数": len(datasets),
            "通過件数": len(result.kept),
            "除外件数": len(result.dropped),
        },
    )
    if len(result.kept) == 0:
        raise ValueError("スクリーニング後に利用可能データが 0 件になりました。閾値を見直してください。")
    return result.kept


def build_dataset_from_parsed(datasets: list[ParsedCTMCDataset], input_is_one_based: bool) -> CTMCSurrogateDataset:
    state_list: list[torch.Tensor] = []
    delta_t_list: list[torch.Tensor] = []
    target_list: list[torch.Tensor] = []

    expected_n: int | None = None

    for ds in datasets:
        if ds.q_mle is None:
            raise ValueError(f"q_mle が None のデータが含まれています: {ds.path}")

        n_state = int(ds.q.shape[0])
        if expected_n is None:
            expected_n = n_state
        elif expected_n != n_state:
            raise ValueError(f"状態数Nが混在しています: expected={expected_n}, got={n_state}, path={ds.path}")

        state_values = ds.samples[:, :2]
        state_min = int(np.min(state_values))
        state_max = int(np.max(state_values))
        if input_is_one_based:
            if state_min < 1 or state_max > n_state:
                raise ValueError(
                    f"状態IDが1始まり前提と整合しません: path={ds.path}, allowed=[1,{n_state}], actual=[{state_min},{state_max}]"
                )
        else:
            if state_min < 0 or state_max > (n_state - 1):
                raise ValueError(
                    f"状態IDが0始まり前提と整合しません: path={ds.path}, allowed=[0,{n_state - 1}], actual=[{state_min},{state_max}]"
                )

        state = torch.as_tensor(state_values.T, dtype=torch.long)
        delta_t = torch.as_tensor(ds.samples[:, 2], dtype=torch.float32)
        target = torch.as_tensor(extract_lambdas_from_Q(ds.q_mle), dtype=torch.float32)

        state_list.append(state)
        delta_t_list.append(delta_t)
        target_list.append(target)

    return CTMCSurrogateDataset(
        state_list=state_list,
        delta_t_list=delta_t_list,
        target_list=target_list,
    )


In [ ]:
validate_paths(model_run_dir, test_data_dir)

model_config_path = model_run_dir / "model_config.yaml"
weights_path = model_run_dir / "weights" / "best_model.pt"
metrics_path = model_run_dir / "metrics.json"

if not model_config_path.exists():
    raise FileNotFoundError(f"model_config が見つかりません: {model_config_path}")
if not weights_path.exists():
    raise FileNotFoundError(f"重みファイルが見つかりません: {weights_path}")

model_config = read_yaml_like_dict(model_config_path)
print("読み込んだ model_config:")
print(model_config)

if metrics_path.exists():
    print("\n学習時 metrics.json:")
    print(json.loads(metrics_path.read_text(encoding="utf-8")))

In [ ]:
parsed = load_dir(test_data_dir, recursive=recursive)
if len(parsed) == 0:
    raise ValueError(f"テストデータが見つかりませんでした: {test_data_dir}")

filtered = apply_eval_screening(parsed) if enable_screening else parsed
if not enable_screening:
    print(f"スクリーニング無効: 入力件数={len(parsed)}")

input_is_one_based = bool(model_config.get("input_is_one_based", True))
test_dataset = build_dataset_from_parsed(filtered, input_is_one_based=input_is_one_based)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=False,
    collate_fn=ctmc_collate_fn,
)

model = build_model(model_config)
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print(f"テストデータ件数: {len(test_dataset)}")


In [ ]:
loss_fn = CustomLoss()

all_preds: list[torch.Tensor] = []
all_targets: list[torch.Tensor] = []
loss_sum = 0.0
total = 0

with torch.no_grad():
    for state, delta_t, target, lengths in test_loader:
        state = state.to(device)
        delta_t = delta_t.to(device)
        target = target.to(device)
        lengths = lengths.to(device)

        pred = model(state, delta_t, lengths)
        loss = loss_fn(pred, target)

        bs = int(state.shape[0])
        loss_sum += float(loss.item()) * bs
        total += bs

        all_preds.append(pred.cpu())
        all_targets.append(target.cpu())

if total == 0:
    raise RuntimeError("テストデータ件数が0件です。")

preds = torch.cat(all_preds, dim=0)
targets = torch.cat(all_targets, dim=0)

mae = torch.mean(torch.abs(preds - targets)).item()
inverse_mae = torch.mean(torch.abs(1.0 / (preds + 1e-12) - 1.0 / (targets + 1e-12))).item()
avg_custom_loss = loss_sum / total

print("=== 検証結果 ===")
print(f"サンプル数: {total}")
print(f"CustomLoss(逆数MAE) 平均: {avg_custom_loss:.6f}")
print(f"MAE(生パラメータ): {mae:.6f}")
print(f"逆数MAE: {inverse_mae:.6f}")

In [ ]:
# 予測値と教師値の先頭数件を確認
preview_count = min(5, preds.shape[0])
for i in range(preview_count):
    print(f"sample[{i}]\n  pred  = {preds[i].numpy()}\n  true  = {targets[i].numpy()}")